In [1]:
import pandas as pd

df = pd.read_csv('../outputs/raw_loaded.csv')

In [2]:
df['NCO_Code'] = df['NCO_Code'].astype(str).str.zfill(3)

print("Starting shape:", df.shape)

Starting shape: (1148634, 153)


In [3]:
flow = {}
flow['Raw file'] = len(df)
print(flow)

{'Raw file': 1148634}


In [7]:
#Filter 1 — First visit only
#df1 = df[df["Visit"] == 'V1'].copy()
df1 = df.loc[df["Visit"].eq("V1")]
flow["First visit only"] = len(df1)
print(flow)

{'Raw file': 1148634, 'First visit only': 1148634}


In [8]:
#Filter 2- Age 15-64
df2 = df1[(df1["Age"] >= 15) & (df1["Age"] <= 64)].copy()
flow["Age 15-64"] = len(df2)
print(flow)

{'Raw file': 1148634, 'First visit only': 1148634, 'Age 15-64': 810208}


In [9]:
#Filter 3 — Regular salaried employment status
df3 = df2[df2["UPSS_Status"] == 31].copy()
flow["Regular salaried (UPSS_Status == 31)"] = len(df3)
print(flow)

{'Raw file': 1148634, 'First visit only': 1148634, 'Age 15-64': 810208, 'Regular salaried (UPSS_Status == 31)': 123541}


In [10]:
#Filter 4 — Non-missing earnings
df4 = df3[df3["Earnings"].notna()].copy()
flow["Non-missing regular earnings"] = len(df4)
print(flow)

{'Raw file': 1148634, 'First visit only': 1148634, 'Age 15-64': 810208, 'Regular salaried (UPSS_Status == 31)': 123541, 'Non-missing regular earnings': 123541}


In [11]:
#Filter 5 — Non-zero earnings
df5 = df4[df4["Earnings"] > 0].copy()
flow["Positive regular earnings"] = len(df5)
print(flow)

{'Raw file': 1148634, 'First visit only': 1148634, 'Age 15-64': 810208, 'Regular salaried (UPSS_Status == 31)': 123541, 'Non-missing regular earnings': 123541, 'Positive regular earnings': 120357}


In [12]:
#Valid occupation code
df6 = df5[df5['NCO_Code'].notna() & (df5['NCO_Code'] != 'nan')].copy()
flow['Valid occupation code'] = len(df6)
print(flow)

{'Raw file': 1148634, 'First visit only': 1148634, 'Age 15-64': 810208, 'Regular salaried (UPSS_Status == 31)': 123541, 'Non-missing regular earnings': 123541, 'Positive regular earnings': 120357, 'Valid occupation code': 120357}


In [13]:
flow_table = pd.DataFrame(list(flow.items()), columns=['Filter Step', 'N Remaining'])
flow_table['% of Raw'] = (flow_table['N Remaining'] / flow_table['N Remaining'].iloc[0] * 100).round(2)
flow_table

,Filter Step,N Remaining,% of Raw
0,Raw file,1148634,100.00
1,First visit only,1148634,100.00
2,Age 15-64,810208,70.54
3,Regular salaried (UPSS_Status == 31),123541,10.76
4,Non-missing regular earnings,123541,10.76
5,Positive regular earnings,120357,10.48
6,Valid occupation code,120357,10.48


In [14]:
df6.to_csv('../outputs/analytical_sample.csv', index=False)
flow_table.to_csv('../outputs/sample_flow_table.csv', index=False)

print("Analytical sample saved:", df6.shape)
print("\nFlow table:")
print(flow_table)

Analytical sample saved: (120357, 153)

Flow table:
                            Filter Step  N Remaining  % of Raw
0                              Raw file      1148634    100.00
1                      First visit only      1148634    100.00
2                             Age 15-64       810208     70.54
3  Regular salaried (UPSS_Status == 31)       123541     10.76
4          Non-missing regular earnings       123541     10.76
5             Positive regular earnings       120357     10.48
6                 Valid occupation code       120357     10.48


In [15]:
print(df['Visit'].dtype)
print(df['Visit'].value_counts())
print(df['Visit'].unique()[:20])

str
Visit
V1    1148634
Name: count, dtype: int64
<StringArray>
['V1']
Length: 1, dtype: str


In [16]:
unique_occ = df6['NCO_Code'].value_counts().reset_index()
unique_occ.columns = ['NCO_3digit', 'Unweighted_Count']

weighted_occ = df6.groupby('NCO_Code')['Weight'].sum().reset_index()
weighted_occ.columns = ['NCO_3digit', 'Weighted_Employment']

occ_summary = unique_occ.merge(weighted_occ, on='NCO_3digit')
occ_summary = occ_summary.sort_values('Weighted_Employment', ascending=False).reset_index(drop=True)

print("Number of unique occupation codes:", occ_summary.shape[0])
occ_summary.head(20)

Number of unique occupation codes: 123


,NCO_3digit,Unweighted_Count,Weighted_Employment
0,522.0,8912,796098362
1,911.0,5646,667867631
2,932.0,5882,628522022
3,541.0,5593,503797321
4,234.0,6517,468850741
5,832.0,4061,387859324
6,833.0,3792,386215942
7,233.0,4714,340507990
8,512.0,3023,338313152
9,411.0,4420,327799527


In [17]:
occ_summary.to_csv('../outputs/occ_summary.csv', index=False)
print("Saved occ_summary.csv")

Saved occ_summary.csv


In [5]:
# Confirm the columns currently available
print(df.columns.tolist())

# Add the renamed fields if the raw names are still present
df = df.rename(columns={
    'pas': 'UPSS_Status',
    'ern_reg': 'Earnings',
    'visit': 'Visit'
})

# Stop early with a clear message if a column is still absent
required = {'Visit', 'UPSS_Status', 'Earnings'}
missing = required - set(df.columns)
assert not missing, f"Missing columns: {missing}"

['File_ID', 'sch', 'qtr', 'month', 'Visit', 'Sector', 'State_Code', 'dc', 'nss_reg', 'bstrm', 'strm', 'grp', 'sstrm', 'sro', 'mfsu', 'sss', 'ssu', 'srl', 'rel', 'Sex', 'Age', 'marst', 'Education_Level', 'tedu_lvl', 'grade', 'yrsbef1', 'yrsaft', 'iflastyr', 'mnths', 'curr_att', 'secondary', 'voc', 'trg', 'trg_com', 'voc_compl', 'voc_fld', 'voc_dur', 'voc_typ', 'voc_fund', 'voc_cert', 'UPSS_Status', 'NIC_Code', 'NCO_Code', 'has_sas', 'loc_pas', 'etyp_pas', 'wrkr_pas', 'job_pas', 'leave_pas', 'ssec_pas', 'ecoprd_pas', 'sas', 'ind_sas', 'ocu_sas', 'loc_sas', 'etyp_sas', 'wrkr_sas', 'job_sas', 'leave_sas', 'ssec_sas', 'ecoprd_sas', 'wrk_365', 'dur_pas', 'dur_sas', 'eff_pas', 'dur_unp', 'evr_wrk', 'rea_nw', 'rea', 'das17', 'ind17', 'hr17', 'ern17', 'das27', 'ind27', 'hr27', 'ern27', 'hr7', 'ahr7', 'das16', 'ind16', 'hr16', 'ern16', 'das26', 'ind26', 'hr26', 'ern26', 'hr6', 'ahr6', 'das15', 'ind15', 'hr15', 'ern15', 'das25', 'ind25', 'hr25', 'ern25', 'hr5', 'ahr5', 'das14', 'ind14', 'hr14', '

In [6]:
# First-visit observations; do not deep-copy 1M+ rows
df1 = df.loc[df['Visit'].eq('V1')]

# Regular salaried workers with positive reported earnings
regular_salaried = df1.loc[
    df1['UPSS_Status'].eq(31) & df1['Earnings'].gt(0)
]

print(regular_salaried['Earnings'].describe())

count    121638.000000
mean      23312.992009
std       21925.484068
min         400.000000
25%       10000.000000
50%       15000.000000
75%       28600.000000
max      500000.000000
Name: Earnings, dtype: float64
